## Autoencoder for Credit Card Fraud Detection
#### By Sawyer Selby

In [ ]:
# Uncomment below line to install if not already on your local device:
# %pip install kaggle keras scikit-learn numpy pandas matplotlib
# Or in the terminal you can do:
# pip install kaggle keras scikit-learn numpy pandas matplotlib

In [ ]:

import os
os.environ["KERAS_BACKEND"] = "torch" # Run Keras on PyTorch. Must be set before importing Keras
from pathlib import Path

import numpy as np # Arrays
import pandas as pd # Tables
import matplotlib.pyplot as plt # Plots

import keras
from keras import layers

from sklearn.model_selection import train_test_split # Train/validation/test split
from sklearn.preprocessing import StandardScaler # Feature scaling
from sklearn.utils.class_weight import compute_class_weight # For class weights
from sklearn.metrics import confusion_matrix, classification_report, f1_score # Evaluation

from kaggle.api.kaggle_api_extended import KaggleApi

import time  # Training and testing time


### Seed

Same seed every run so the split, starting weights and dropout don't change between runs.

In [ ]:
SEED = 42
keras.utils.set_random_seed(SEED) # Same random numbers every run

### Setup

Gets the dataset from Kaggle. If the CSV is already downloaded it skips it.

In [ ]:
DATA_DIR = "../data/"  # Folder for Kaggle data
DATASET = "mlg-ulb/creditcardfraud"  # Kaggle dataset ID (owner/dataset-name)

tokenPath = Path.home() / ".kaggle" / "access_token"
os.environ["KAGGLE_API_TOKEN"] = tokenPath.read_text().strip()

os.makedirs(DATA_DIR, exist_ok=True)  # Creates data folder

api = KaggleApi()
api.authenticate()  # Needs the API token in .kaggle/access_token in your home folder

# Downloads and unzips the dataset, skips if the CSV is already there
csvPath = os.path.join(DATA_DIR, "creditcard.csv")
if not os.path.exists(csvPath):
    api.dataset_download_files(DATASET, path=DATA_DIR, unzip=True)


print(os.listdir(DATA_DIR))

### Checking Data

Looking at the shape, columns, how many of each class, missing values and duplicates.

Class is the dependent variable for whether a transaction was fraudulent (1) or real (0).

Dropped the duplicate rows so the same transaction can't be in both train and test.

In [ ]:
df = pd.read_csv("../data/creditcard.csv")
print(df.head()) # First 5 rows
print(df.shape) # Rows, columns
print(df.columns.tolist()) # Column names
print(df["Class"].value_counts()) # 0s vs 1s (legit vs fraud)
print(df.isnull().sum().sum()) # Total missing values
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True) # Drop duplicate rows

### Labels

Class counts after dropping duplicates.

In [ ]:
counts = df["Class"].value_counts() # Real vs fraud count
print(f"{df['Class'].nunique()} classes, {counts.min()} to {counts.max()}")

### Train/Test/Validation Split

70% train, 15% validation, 15% test. Stratify keeps the fraud ratio the same in all three.

Validation is for early stopping and picking the threshold. Test only gets used at the end.

In [ ]:
# 15% test
trainValSplit, testSplit = train_test_split(
    df,
    test_size = 0.15,
    stratify=df["Class"], # Keeps the fraud ratio the same
    random_state = SEED,
)
# The rest is split into 70% training and 15% validation
trainSplit, valSplit = train_test_split(
    trainValSplit,
    test_size = 0.15 / 0.85, # 15% of the full dataset
    stratify = trainValSplit["Class"],
    random_state = SEED,
)

# Separate features from the label
xTrain = trainSplit.drop(columns=["Class"])
yTrain = trainSplit["Class"].values

xVal = valSplit.drop(columns=["Class"])
yVal = valSplit["Class"].values

xTest = testSplit.drop(columns=["Class"])
yTest = testSplit["Class"].values

### Scaling

Scaling all 30 features. Amount can be in the thousands, and even though V1 - V28 are PCA components they don't all have the same spread, so the bigger ones would dominate training and the autoencoder's MSE.

fit_transform on train, transform on validation and test so nothing from them leaks into training.

In [ ]:
colsToScale = list(xTrain.columns) # All 30 features
scaler = StandardScaler()

# Make copies
xTrain = xTrain.copy()
xVal = xVal.copy()
xTest = xTest.copy()

xTrain[colsToScale] = scaler.fit_transform(xTrain[colsToScale]) # Learns the mean and standard deviation of each feature

# Transform val and test using training mean/standard deviations
xVal[colsToScale] = scaler.transform(xVal[colsToScale])
xTest[colsToScale] = scaler.transform(xTest[colsToScale])

print(xTrain[colsToScale].describe().loc[["mean", "std"]].round(2))
print(xTest[colsToScale].describe().loc[["mean", "std"]].round(2))

# Converting to NumPy float32 before training for Keras
xTrainArr = xTrain.to_numpy(dtype="float32")
xValArr = xVal.to_numpy(dtype="float32")
xTestArr = xTest.to_numpy(dtype="float32")

### Class Weights

Fraud is only about 0.17% of the data so the model could just ignore it and still get high accuracy.

This makes fraud mistakes cost more in the loss. Computes weights once from the training labels only then passes them to training.

Using the same weights on the validation loss so both losses are measured the same way.

In [ ]:
classes = np.array([0, 1])
weights = compute_class_weight(class_weight = "balanced", classes = classes, y = yTrain) # Rarer class gets the bigger weight

classWeights = {0: weights[0], 1: weights[1]}
print("Training Count:", np.bincount(yTrain))
print("Class Weights:", classWeights)

classWeights = {0: float(weights[0]), 1: float(weights[1])} # Keras wants plain floats
valSampleWeight = np.where(yVal == 1, classWeights[1], classWeights[0]) # Weight for each validation row

### Baseline (Feedforward Network)

FNN since it's the standard baseline for tabular data.

##### Build baseline

30 -> 64 -> 32 -> 1
Dropout(0.3) randomly switches off 30% of the neurons during each training step, helping prevent overfitting.

Sigmoid output is a fraud probability between 0 and 1.

Precision and recall are tracked during training to see whether the model catches any fraud.

In [ ]:
def buildBaseline(nFeatures):
    model = keras.Sequential([
        layers.Input(shape=(nFeatures,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),  # Randomly drops 30% of neurons each step to reduce overfitting
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),  # Fraud probability (0 to 1)
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy", # Loss for binary classification
        metrics=["accuracy",
                 keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall")],
    )
    return model

baseline = buildBaseline(xTrainArr.shape[1]) # 30 input features
baseline.summary()

##### Train with early stopping and timing

Early stopping watches validation loss. If it doesn't improve for 5 epochs, training stops and the best weights are restored.

Batch size 2048 because fraud is so rare that small batches often contain no fraud at all. Large batches make it more likely that each one has a few.

time.perf_counter() around fit gives training time.

In [ ]:
RESULTS_DIR = "../results/"
os.makedirs(RESULTS_DIR, exist_ok=True) # Folder for saved plots

earlyStop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,  # Stop after 5 epochs with no improvement
    restore_best_weights=True,  # Roll back to the best epoch
)

start = time.perf_counter()
baselineHistory = baseline.fit(
    xTrainArr, yTrain.astype("float32"),
    validation_data=(xValArr, yVal.astype("float32"), valSampleWeight), # Weighted validation loss
    class_weight=classWeights, # Weighted training loss
    epochs=50,
    batch_size=2048,  # Large batches so most contain some fraud
    callbacks=[earlyStop],
    verbose=1,
)
baselineTrainTime = time.perf_counter() - start
print(f"Baseline training time: {baselineTrainTime:.1f} s, "
      f"epochs run: {len(baselineHistory.history['loss'])}")

##### Training curves

Train vs validation loss and accuracy. If validation loss goes up while training loss keeps going down then it's overfitting, and early stopping goes back to the best epoch.

Accuracy stays high the whole time since almost everything is real, so loss is the better one to look at for overfitting.

In [ ]:
h = baselineHistory.history # Loss and metrics for each epoch
fig, axes = plt.subplots(1, 2, figsize=(12, 4)) # Loss on the left, accuracy on the right

axes[0].plot(h["loss"], label="Train")
axes[0].plot(h["val_loss"], label="Validation")
axes[0].set_title("Baseline FNN: Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(h["accuracy"], label="Train")
axes[1].plot(h["val_accuracy"], label="Validation")
axes[1].set_title("Baseline FNN: Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "baseline_curves.png"), dpi=150)
plt.show()

##### Test time

How long it takes to predict the whole test set.

0.5 threshold is just a preview, the final results use the tuned threshold.

In [ ]:
start = time.perf_counter()
yProbBase = baseline.predict(xTestArr, batch_size=4096).ravel()  # Fraud probabilities
baselineTestTime = time.perf_counter() - start
print(f"Baseline test time: {baselineTestTime:.2f} s")

# Preview at the default 0.5 threshold
yPredBase = (yProbBase >= 0.5).astype(int) # 1 = fraud, 0 = real
print(classification_report(yTest, yPredBase, digits=4, target_names=["Real", "Fraud"]))
print(confusion_matrix(yTest, yPredBase)) # Rows = actual, columns = predicted

### Autoencoder

Only trained on real transactions so it learns what normal looks like. Fraud should get rebuilt badly.

Encoder: 30 -> 20 -> 10. Squeezes each transaction down to 10 numbers (the bottleneck).

Decoder: 10 -> 20 -> 30. Rebuilds the transaction from those 10 numbers.

Loss is MSE between the original and rebuilt transaction. Added min_delta so it stops once the loss isn't really improving anymore.

Batch size 2048 to cut training time. With batch size 256 it took about 200 s, now it's about 34 s. It ran all 100 epochs since the validation loss was still going down a little.

In [ ]:
xTrainReal = xTrainArr[yTrain == 0]  # Real transactions only
xValReal = xValArr[yVal == 0]

encoder = keras.Sequential([
    layers.Input(shape=(30,)),
    layers.Dense(20, activation="relu"),
    layers.Dense(10, activation="relu"),  # 10-number summary
])
decoder = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(20, activation="relu"),
    layers.Dense(30),  # No activation because features can be negative
])
autoencoder = keras.Sequential([encoder, decoder]) # Encoder then decoder
autoencoder.compile(optimizer="adam", loss="mse")

aeEarlyStop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, min_delta=1e-4, restore_best_weights=True)

start = time.perf_counter()
aeHistory = autoencoder.fit(
    xTrainReal, xTrainReal,  # Input and target are the same
    validation_data=(xValReal, xValReal),
    epochs=100,
    batch_size=2048,
    callbacks=[aeEarlyStop],
    verbose=2, # One line per epoch
)
aeTrainTime = time.perf_counter() - start
print(f"Autoencoder training time: {aeTrainTime:.1f} s")

In [ ]:
# Only loss, autoencoder doesn't have accuracy
plt.plot(aeHistory.history["loss"], label="Train")
plt.plot(aeHistory.history["val_loss"], label="Validation")
plt.title("Autoencoder: Loss")
plt.xlabel("Epoch")
plt.legend()
plt.savefig(os.path.join(RESULTS_DIR, "autoencoder_loss.png"), dpi=150)
plt.show()

### New Features from the Autoencoder

Reconstruction error is the average squared difference between the transaction and the rebuilt one. Higher error means it doesn't look normal.

The errors are really skewed so I used log1p on them, then scaled it (fit on training only).

The error gets added onto the 30 original features, so the classifier gets 31 inputs.

In [ ]:
def aeFeatures(x):
    summary = encoder.predict(x, batch_size=4096, verbose=0) # 10 numbers per transaction
    error = np.mean((x - autoencoder.predict(x, batch_size=4096, verbose=0)) ** 2, axis=1) # Reconstruction error
    return np.column_stack([summary, np.log1p(error)]) # 10 + 1 = 11 columns

featScaler = StandardScaler()
newTrain = np.column_stack([xTrainArr, featScaler.fit_transform(aeFeatures(xTrainArr)[:, -1:])]) # 30 original + 1 error = 31
newVal = np.column_stack([xValArr, featScaler.transform(aeFeatures(xValArr)[:, -1:])])

print("Real log-error (median): ", np.median(aeFeatures(xValArr)[yVal == 0, -1]))
print("Fraud log-error (median):", np.median(aeFeatures(xValArr)[yVal == 1, -1]))

### Proposed Model

Same FNN, class weights, batch size and early stopping as the baseline. Only difference is the input: the 30 original features plus the reconstruction error (31 total).

Versions I tried:
1. Only the 11 autoencoder features (encoder's 10 numbers + error). Did worse than the baseline (F1 0.6512 vs 0.7383) since 10 numbers can't hold everything the 30 features do.
2. 30 original + 11 autoencoder features (41). Slightly better than the baseline (mean test F1 0.7955 vs 0.7897 over 5 seeds), but took 209 s to train.
3. 30 original + only the error (31), with all 30 features scaled and autoencoder batch size 2048. This is the current version.

Training time includes the autoencoder since the model needs it to run.

In [ ]:
proposed = buildBaseline(newTrain.shape[1]) # Same architecture, 31 inputs

start = time.perf_counter()
proposedHistory = proposed.fit(
    newTrain, yTrain.astype("float32"),
    validation_data=(newVal, yVal.astype("float32"), valSampleWeight),
    class_weight=classWeights,
    epochs=50,
    batch_size=2048,
    callbacks=[earlyStop], # Same early stopping as the baseline
    verbose=2,
)
proposedTrainTime = aeTrainTime + (time.perf_counter() - start)  # Autoencoder + classifier
print(f"Proposed training time (autoencoder + classifier): {proposedTrainTime:.1f} s")

In [ ]:
h = proposedHistory.history # Loss and metrics for each epoch
fig, axes = plt.subplots(1, 2, figsize=(12, 4)) # Loss on the left, accuracy on the right

axes[0].plot(h["loss"], label="Train")
axes[0].plot(h["val_loss"], label="Validation")
axes[0].set_title("Proposed Model: Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(h["accuracy"], label="Train")
axes[1].plot(h["val_accuracy"], label="Validation")
axes[1].set_title("Proposed Model: Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "proposed_curves.png"), dpi=150)
plt.show()

In [ ]:
# Test time includes making the autoencoder features
start = time.perf_counter()
newTest = np.column_stack([xTestArr, featScaler.transform(aeFeatures(xTestArr)[:, -1:])]) # Same scaler as training
yProbProp = proposed.predict(newTest, verbose=0).ravel() # Fraud probabilities
proposedTestTime = time.perf_counter() - start
print(f"Proposed test time: {proposedTestTime:.2f} s")

### Threshold Tuning

At 0.5 both models flag way too many real transactions as fraud.

Trying thresholds from 0.900 to 0.999 on the validation set and keeping whichever gives the best fraud F1. The models are really confident so the best threshold ends up close to 1. Each model gets its own threshold. Test set isn't used here.

In [ ]:
# Validation probabilities
yValProbBase = baseline.predict(xValArr, verbose=0).ravel()
yValProbProp = proposed.predict(newVal, verbose=0).ravel()

thresholds = np.arange(0.90, 1.00, 0.001)
baseF1s = [f1_score(yVal, (yValProbBase >= t).astype(int)) for t in thresholds] # Fraud F1 at each threshold
propF1s = [f1_score(yVal, (yValProbProp >= t).astype(int)) for t in thresholds]

baseThreshold = thresholds[np.argmax(baseF1s)] # Threshold with the highest F1
propThreshold = thresholds[np.argmax(propF1s)]
print(f"Baseline threshold: {baseThreshold:.2f}")
print(f"Proposed threshold: {propThreshold:.2f}")

### Final Test Results

Both models on the test set with their tuned thresholds.

Fraud row precision, recall and F1 are what matter. Accuracy is high either way since almost every transaction is real.

In [ ]:
# Apply each model's tuned threshold
yPredBase = (yProbBase >= baseThreshold).astype(int)
yPredProp = (yProbProp >= propThreshold).astype(int)

print(f"BASELINE (threshold {baseThreshold:.2f})")
print(classification_report(yTest, yPredBase, digits=4, target_names=["Real", "Fraud"]))
print(confusion_matrix(yTest, yPredBase)) # Rows = actual, columns = predicted

print(f"\nPROPOSED (threshold {propThreshold:.2f})")
print(classification_report(yTest, yPredProp, digits=4, target_names=["Real", "Fraud"]))
print(confusion_matrix(yTest, yPredProp))

# Training and testing time
print(f"\nBaseline: train {baselineTrainTime:.1f} s, test {baselineTestTime:.2f} s")
print(f"Proposed: train {proposedTrainTime:.1f} s, test {proposedTestTime:.2f} s")

### Seed Check

Retraining both classifiers with 5 different seeds to see if the improvement is the same. Same autoencoder each time, threshold tuned on validation each time.

Validation F1 is what I used to compare versions, test F1 is the final result.

In [ ]:
seedF1 = {"Baseline": [], "Proposed": []}
seedVal = {"Baseline": [], "Proposed": []} # Validation F1, used to compare versions
for s in [1, 2, 3, 4, 5]:
    keras.utils.set_random_seed(s)
    for name, xTr, xV, xTe in [("Baseline", xTrainArr, xValArr, xTestArr),
                               ("Proposed", newTrain, newVal, newTest)]:
        model = buildBaseline(xTr.shape[1])
        model.fit(xTr, yTrain.astype("float32"),
                  validation_data=(xV, yVal.astype("float32"), valSampleWeight),
                  class_weight=classWeights, epochs=50, batch_size=2048,
                  callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
                  verbose=0)
        valProb = model.predict(xV, verbose=0).ravel()
        valF1s = [f1_score(yVal, (valProb >= t).astype(int)) for t in thresholds]
        bestT = thresholds[np.argmax(valF1s)] # Tuned on validation
        seedVal[name].append(max(valF1s))
        testProb = model.predict(xTe, verbose=0).ravel()
        seedF1[name].append(f1_score(yTest, (testProb >= bestT).astype(int)))

for name in seedF1:
    print(f"{name}: mean val F1 {np.mean(seedVal[name]):.4f} | mean test F1 {np.mean(seedF1[name]):.4f} ± {np.std(seedF1[name]):.4f}")

### Literature Comparison

Hardware: AMD Ryzen 7 9850X3D

| Source | Precision | Recall | F1 | Train time | Test time | Notes |
|---|---|---|---|---|---|---|
| Baseline | 0.8182 | 0.7606 | 0.7883 (mean 0.7911 over 5 seeds) | 5.5 s | 0.02 s | |
| Proposed | 0.7887 | 0.7887 | 0.7887 (mean 0.8000 over 5 seeds) | 38.7 s | 1.17 s | Includes autoencoder training |
| Ma & Hargreaves 2025, AAE | 0.8295 | 0.8008 | 0.8149 | not reported | not reported | Split not reported |
| Ma & Hargreaves 2025, plain AE | 0.322 | 0.791 | 0.458 | not reported | not reported | |
| Adejoh et al. 2025, AE-ASOM | - | - | 0.967* | 200.52 s | not reported | *Not fraud-class F1, can't compare directly |
| Adejoh et al. 2025, RBM-ASOM | - | - | 0.955* | 199.82 s | 4.33 s | Intel i7-10700, Quadro P620 |